In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import jax

import jax.numpy as jnp
import equinox as eqx

from admpe.data_management import load_excitation_result, list_excitation_results
from admpe.modeling.gp_modeling import (
    GPTransitionModel,
    fit_gp_transitions,
)
from admpe.excitation.excitation_experiment_setup import setup_environment

In [ ]:
# load train data
system_name = "cart_pole"
exp_name = "state_only_pm_no_noise"
# exp_name = "pm_no_noise"
excitation_results_paths = list_excitation_results(
    system_name=system_name,
    exp_name=exp_name,
    excitation_objective="dmpe_static_LOLA_ls",
)

train_data = load_excitation_result(excitation_results_paths[0]["path"])

# loading the data sets precision to the data precision
jax.config.update("jax_enable_x64", True)

In [ ]:
system_name = train_data["metadata"]["system_name"]
env, _, featurize, _, _ = setup_environment(system_name, "neural_euler")


n = 6999
observations = train_data["observations"][:n + 1].astype(jnp.float64)
actions = train_data["actions"][:n].astype(jnp.float64)

model = GPTransitionModel(
    obs_dim=observations.shape[-1],
    action_dim=actions.shape[-1],
    n_train_points=n,
    featurize=featurize,
    angle_indices={
        "fluid_tank": (),
        "pendulum": (0,),
        "cart_pole": (2,),
    }[system_name],
)

print("Sucessfully created model")
print(jax.config.jax_enable_x64)
print(observations.dtype, actions.dtype, model.train_x.dtype)

model, histories = fit_gp_transitions(
    model,
    observations,
    actions,
    max_iters=3,
)

prediction = model.step(observations[0], actions[0], tau=1.0)

In [ ]:
prediction = model.step(observations[0], actions[0], tau=1.0)
prediction

In [ ]:
from admpe.evaluation.model_wrapping import NodeModelWrapper, EnvWrapper
from admpe.evaluation.modeling_visualization import visualize_model_prediction

In [ ]:
wrapped_model = NodeModelWrapper(
    model=model,
    featurize=featurize,
)


env = eqx.tree_at(lambda s: s.process_noise_variance, env, 0.0)
wrapped_env = EnvWrapper(
    model=env,
    featurize=featurize,
)

In [ ]:
visualize_model_prediction(20, wrapped_model, wrapped_env)

In [ ]:
training_points = np.concatenate(
    [
        np.asarray(train_data["observations"])[:-1],
        np.asarray(train_data["actions"]),
    ],
    axis=-1,
)

fig, axs = visualize_model_prediction(20, wrapped_model, wrapped_env)
for j in range(training_points.shape[1]):
    for i in range(training_points.shape[1]):
        if i == j:
            continue

        axs[j, i].scatter(
            training_points[:, i],
            training_points[:, j],
            s=2,
            color="red",
            alpha=0.25,
            linewidths=0,
            zorder=3,
            rasterized=True,
        )

plt.show()